## 1. Imports and Setup

In [1]:
from builtins import int, float
import pandas as pd
import requests
from datetime import datetime
import os

# Ensure data folder exists
os.makedirs('../data', exist_ok=True)


## 2. Historical Data Ingestion (API or CSV Source)

In [2]:
hist_path = '../data/bitcoin_historical_cleaned.csv'

if not os.path.exists(hist_path):
    print("📥 Ingesting historical data from 1-min CSV...")

    df = pd.read_csv('../data/btcusd_1-min_data.csv')
    df['timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
    df['price_usd'] = df['Close']
    df = df[['timestamp', 'price_usd']]
    df.to_csv(hist_path, index=False)

    print(f"✅ Saved historical data: {len(df)} rows.")
else:
    print(f"✅ Skipping — historical file already exists at {hist_path}")

✅ Skipping — historical file already exists at ../data/bitcoin_historical_cleaned.csv


## 3. Live Price Ingestion

In [3]:
live_path = '../data/bitcoin_live.csv'

url = 'https://api.coingecko.com/api/v3/simple/price'
params = {'ids': 'bitcoin', 'vs_currencies': 'usd'}

try:
    r = requests.get(url, params=params)
    price = float(r.json()['bitcoin']['usd'])
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    df_live = pd.DataFrame([{'timestamp': timestamp, 'price_usd': price}])

    if os.path.exists(live_path):
        df_live.to_csv(live_path, mode='a', header=False, index=False)
        print(f"✅ Appended live price: {price} USD")
    else:
        df_live.to_csv(live_path, index=False)
        print(f"✅ Created live price file: {price} USD")
except Exception as e:
    print(f"❌ Live price fetch failed: {e}")

✅ Appended live price: 103256.0 USD
